###install Ba

In [1]:
!pip install -qU langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface faiss-cpu sentence-transformers groq pydantic pypdf gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9

### IMPORTS LIB

In [2]:
import os
import uuid
import time
from datetime import datetime
import random
import logging
from pathlib import Path
import re
import json

import torch
import gradio as gr

from google.colab import files, userdata
from groq import Groq
from langchain_groq import ChatGroq

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS ,DistanceStrategy

from langchain_community.chat_message_histories import ChatMessageHistory

/tmp/ipykernel_23465/658384758.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### LOAD GROQ API KEY SECURELY


In [3]:
GROQ_API_KEY = userdata.get("groq")

if not GROQ_API_KEY:
    raise ValueError("GROQ API KEY not found in Colab Secrets")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print(" GROQ API KEY Loaded Successfully Tmaam")

 GROQ API KEY Loaded Successfully Tmaam


###uploade files

In [4]:
uploaded = files.upload()
text = list(uploaded.keys())[0]
print("Uploaded:", text)

Saving crocoooo.pdf to crocoooo.pdf
Uploaded: crocoooo.pdf


In [5]:
loader = PyPDFLoader(text)
documents = loader.load()

###Show All Pages

In [6]:
for i, doc in enumerate(documents):

    print(".." * 50)
    print(f"PAGE: {i+1}")
    print(f"SOURCE: {doc.metadata}")
    print(".." * 50)
    print(doc.page_content[:1500])
    print("\n")

....................................................................................................
PAGE: 1
SOURCE: {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-06-15T03:15:41+03:00', 'author': 'ismail elsayed eltanja', 'moddate': '2026-06-15T03:15:41+03:00', 'source': 'crocoooo.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}
....................................................................................................
Croco IT – Structured Company 
This document is a professionally organized English knowledge base about Croco IT designed 
specifically for Retrieval-Augmented Generation (RAG) systems. 
The content is cleaned, expanded, and structured for semantic search, embeddings, chunking, 
and vector databases. 
Official Website: Croco IT Official Website 
 
Company Overview 
Croco IT is a fast-growing software house and digital transformation company specializing in 
enterprise software development, ecommerce solutions

###Tokenization , Chunking

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/multilingual-e5-base")

def clean_text(text):
    text = re.sub(r'\s+',' ', text)
    return text.strip()

def token_len(text):
    return len(tokenizer.encode(text))

random.seed(101)
random.shuffle(documents)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
    length_function=token_len,
    separators=["\n# ","\n## ","\n### ","\n\n","\n",". ","؟ ","! ","، "," ",""])

chunks = splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
print(chunks[0].page_content) #1

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Number of chunks: 16
Consultation Booking https://appointments.crocoit.com 
LinkedIn https://www.linkedin.com/company/crocoit 
Twitter / X https://x.com/CrocoIt 
Facebook https://www.facebook.com/CrocoIT/ 
Privacy Policy https://crocoit.com/privacy-policy-2/ 
Address   : 501 Silverside Road, Suite 92 
Wilmington, DE 19809 
United States 
Mobile   : +1 302-404-6080 
Email Address 
info@crocoit.com 
Official Website 
https://crocoit.com/


###Embeddings

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device is: {device}")
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True} )

Using device is: cpu


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

###Vectorstore


In [9]:
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i

if os.path.exists("faiss_index"):
    vectorstore = FAISS.load_local(
        "faiss_index",
        embedding_model,
        allow_dangerous_deserialization=True)
else:
    vectorstore = FAISS.from_documents(
              documents=chunks,
              embedding=embedding_model,
   distance_strategy=DistanceStrategy.COSINE)

vectorstore.save_local("faiss_index")

###retriever

In [10]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 10,
        "lambda_mult": 0.5})

###Groq client

In [11]:
client = Groq(
    api_key=os.environ["GROQ_API_KEY"])

###Chat_Message_History

In [22]:
memory_store = {}
WINDOW_SIZE = 8

def get_history(session_id: str):
    if session_id not in memory_store:
        memory_store[session_id] = ChatMessageHistory()
    return memory_store[session_id]

def get_recent_messages(session_id: str):
    history = get_history(session_id)
    messages = history.messages[-WINDOW_SIZE * 2:]
    formatted_messages = []

    for msg in messages:
        if msg.type == "human":
            formatted_messages.append({
                "role": "user",
                "content": msg.content
            })
        elif msg.type == "ai":
            formatted_messages.append({
                "role": "assistant",
                "content": msg.content
            })

    return formatted_messages

def ask(question: str, session_id: str = "default"):
    try:
        question = question.strip()

        if not question:
            return "Please enter a question."

        search_query = f"query: {question}"

        docs = retriever.invoke(search_query)

        if not docs:
            return "I don't have information about this question"

        context_parts = []

        for i, doc in enumerate(docs, start=1):
            chunk = doc.page_content.replace("passage:", "").strip()
            source = doc.metadata.get("source", "")

            if source:
                context_parts.append(
                    f"[Document {i}]\nSource: {source}\n{chunk}"
                )
            else:
                context_parts.append(
                    f"[Document {i}]\n{chunk}"
                )

        context = "\n\n".join(context_parts)

        history = get_history(session_id)

        messages = [{
            "role": "system",
            "content": """

You are a professional AI assistant representing Croco IT.

Your role:
You are the official Croco IT assistant. You help users understand the company’s services, products, and technologies in a clear, accurate, and professional way.

Core Rules:
1. Use ONLY the information provided in the Knowledge Base Context.
2. Never guess, infer, or invent any information.
3. If the answer is not explicitly found in the context, respond exactly:
   "I don't have information about this question"
4. Never mention or refer to:
   - Context
   - Documents
   - Chunks
   - Knowledge Base
   - Internal processing or system behavior

Language Rule:
- Always respond in the SAME language used by the user.

Answer Style:
- Start with a clear direct answer.
- Then provide supporting details if needed.
- Keep responses concise, structured, and professional.
- Use Markdown formatting when helpful.
- Use bullet points for lists.

Services / Products Explanations:
When relevant, include:
- Description
- Benefits
- Technologies (ONLY if explicitly present in context)

Conversation Behavior:
- If the user sends greetings (e.g. "hello", "hi", "السلام عليكم"):
  Respond politely in a professional tone.
- If the message is casual and unrelated to Croco IT:
  Respond politely, then gently guide the conversation toward Croco IT services.
- If the question is unclear:
  Ask a short clarification question.

Strict Restrictions:
- Do NOT repeat phrases unnecessarily.
- Do NOT provide unverified information.
- Do NOT create fake links, services, or claims.
- Do NOT express personal opinions.

Scope Control:
- If the question is outside Croco IT scope, respond exactly:
  "This question is outside the scope of the information available about Croco IT"

Memory Usage:
- You may use conversation history ONLY to maintain context of the conversation flow.
- Never use it as a factual source.

Goal:
Provide accurate, professional, and human-like support responses strictly based on the provided knowledge base.


"""}]

        # ADD CHAT HISTORY
        history_messages = get_recent_messages(session_id)

        if history_messages:
            messages.extend(history_messages)

        # CURRENT QUESTION
        messages.append({
            "role": "user",
            "content": f"""
Knowledge Base Context:{context}
Current User Question:{question}
Remember:
Answer only from the Knowledge Base Context.
If information is missing, respond exactly:
I don't have information about this question
"""})

        # LLM
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0.1,
            max_tokens=601,
            messages=messages)

        answer = response.choices[0].message.content.strip()

        # SAVE MEMORY
        if answer:
            history.add_user_message(question)
            history.add_ai_message(answer)

        return answer

    except Exception as e:
        return f"Error: {str(e)}"

###GUI

In [21]:
# AUTH (Colab Secrets + Local fallback)
try:
    VALID_USERNAME = userdata.get("VALID_USERNAME")
    VALID_PASSWORD = userdata.get("VALID_PASSWORD")

    if not VALID_USERNAME:
        VALID_USERNAME = os.getenv("VALID_USERNAME")

    if not VALID_PASSWORD:
        VALID_PASSWORD = os.getenv("VALID_PASSWORD")

except Exception:
    VALID_USERNAME = os.getenv("VALID_USERNAME")
    VALID_PASSWORD = os.getenv("VALID_PASSWORD")


# MEMORY SYSTEM
memory_store = {}
WINDOW_SIZE = 8


def get_history(session_id: str):
    if session_id not in memory_store:
        memory_store[session_id] = ChatMessageHistory()
    return memory_store[session_id]


def get_recent_messages(session_id: str):
    history = get_history(session_id)
    messages = history.messages[-WINDOW_SIZE * 2:]

    out = []
    for msg in messages:
        role = "user" if msg.type == "human" else "assistant"
        out.append({"role": role, "content": msg.content})

    return out

def clear_chat(session_id):
    if session_id in memory_store:
        memory_store[session_id] = ChatMessageHistory()
    return [], session_id


# LOGIN
def login(u, p):
    if u == VALID_USERNAME and p == VALID_PASSWORD:
        sid = str(uuid.uuid4())

        return (
            gr.update(visible=False),   # login panel
            gr.update(visible=True),    # chat panel
            sid,
            [],
            f"Session: {sid[:8]}"
        )

    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "",[]," Invalid Username Or Password"
    )


# CHAT WRAPPER → RAG
def chat_fn(message, history, session_id):
    if not session_id:
        return history, ""

    answer = ask(message, session_id)

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    return history, ""

# UI
with gr.Blocks(theme=gr.themes.Soft(), title="Croco IT RAG System") as demo:

    session_state = gr.State("")

    login_panel = gr.Column(visible=True)
    chat_panel = gr.Column(visible=False)

    with login_panel:
        gr.Markdown(" Croco IT Login System")

        username = gr.Textbox(label="Username")
        password = gr.Textbox(label="Password", type="password")

        login_btn = gr.Button("Login", variant="primary")
        login_status = gr.Textbox(label="Status", interactive=False)

    with chat_panel:
        gr.Markdown("Croco IT AI Assistant")

        session_box = gr.Textbox(label="Session ID", interactive=False)

        chatbot = gr.Chatbot(height=450)

        with gr.Row():
            msg = gr.Textbox(label="Message", scale=8)
            send = gr.Button("Send", variant="primary", scale=2)

        clear = gr.Button("Clear Chat")

    # login
    login_btn.click(
        login,
        [username, password],
        [login_panel, chat_panel, session_state, chatbot, login_status]
    ).then(
        lambda sid: sid,
        session_state,
        session_box)
    # send
    send.click(
        chat_fn,
        [msg, chatbot, session_state],
        [chatbot, msg])

    msg.submit(
        chat_fn,
        [msg, chatbot, session_state],
        [chatbot, msg])

    # clear
    clear.click(
        clear_chat,
        [session_state],
        [chatbot, session_state])

demo.launch()

/tmp/ipykernel_23465/4230669585.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Croco IT RAG System") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://837b57c5fc9b377505.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
